## Avg metrics for Baselines

In [ ]:
# ---- output capture (shared CSV) ----
from pathlib import Path
import csv

OUTPUT_CSV = Path('eval/statistics_cell_outputs.csv')
CSV_COLUMNS = [
    'section',
    'user',
    'baseline',
    'category',
    'mean_llm_score',
    'num_scores',
    'num_files',
    'used_summary_mean_fallback',
    'eval_file',
]


def append_rows_to_csv(rows, section):
    if not rows:
        return
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    file_exists = OUTPUT_CSV.exists()
    with OUTPUT_CSV.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            record = {k: None for k in CSV_COLUMNS}
            record.update({'section': section})
            record.update(row)
            writer.writerow(record)

from pathlib import Path
import json
import statistics

# llm_scores are taken from per-sample scores["llm_gpt_score"] in eval JSONs
root = Path('generation')

baseline_files = {
    'oracle': [root / 'oracle' / 'results'],
    'hipporag2_top5': [root / 'HippoRAG2' / 'results'],
    'memoryos': [root / 'MemoryOS' / 'results'],
    'rag_top5': [root / 'rag' / 'results'],
    'rag_top10': [root / 'rag' / 'results'],
    'rag_top20': [root / 'rag' / 'results'],
}

filename_map = {
    'oracle': 'oracle_results_eval.json',
    'hipporag2_top5': 'hipporag2_results_top5_eval.json',
    'memoryos': 'memoryos_results_eval.json',
    'rag_top5': 'rag_results_top5_eval.json',
    'rag_top10': 'rag_results_top10_eval.json',
    'rag_top20': 'rag_results_top20_eval.json',
}


def iter_eval_files(baseline: str):
    name = filename_map[baseline]
    for base in baseline_files[baseline]:
        if base.exists():
            yield from base.rglob(f'**/eval/{name}')


def load_llm_scores(path: Path):
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    scores = []
    for sample in data.get('samples', []):
        score = sample.get('scores', {}).get('llm_gpt_score')
        if isinstance(score, (int, float)):
            scores.append(float(score))
    if scores:
        return scores, False
    # Fallback to summary mean if per-sample scores missing
    mean_score = data.get('summary', {}).get('llm_gpt_score_mean')
    if isinstance(mean_score, (int, float)):
        return [float(mean_score)], True
    return [], False


rows = []
for baseline in ['oracle', 'rag_top5', 'rag_top10', 'rag_top20', 'memoryos', 'hipporag2_top5']:
    all_scores = []
    used_fallback = False
    files = sorted(iter_eval_files(baseline))
    for fp in files:
        scores, used_mean = load_llm_scores(fp)
        if used_mean:
            used_fallback = True
        all_scores.extend(scores)
    mean = statistics.mean(all_scores) if all_scores else float('nan')
    rows.append({
        'baseline': baseline,
        'mean_llm_score': mean,
        'num_scores': len(all_scores),
        'num_files': len(files),
        'used_summary_mean_fallback': used_fallback,
    })
append_rows_to_csv(rows, 'avg_metrics')

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
except Exception:
    rows

output_path = Path('eval/baseline_overview.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)
print(f"Saved results to {output_path}")

## User Level Scores

In [ ]:
# ---- output capture (shared CSV) ----
from pathlib import Path
import csv

OUTPUT_CSV = Path('eval/statistics_cell_outputs.csv')
CSV_COLUMNS = [
    'section',
    'user',
    'baseline',
    'category',
    'mean_llm_score',
    'num_scores',
    'num_files',
    'used_summary_mean_fallback',
    'eval_file',
]


def append_rows_to_csv(rows, section):
    if not rows:
        return
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    file_exists = OUTPUT_CSV.exists()
    with OUTPUT_CSV.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            record = {k: None for k in CSV_COLUMNS}
            record.update({'section': section})
            record.update(row)
            writer.writerow(record)

from collections import defaultdict
from pathlib import Path
import json
import statistics

root = Path('generation')

baseline_files = {
    'oracle': [root / 'oracle' / 'results'],
    'hipporag2_top5': [root / 'HippoRAG2' / 'results'],
    'memoryos': [root / 'MemoryOS' / 'results'],
    'rag_top5': [root / 'rag' / 'results'],
    'rag_top10': [root / 'rag' / 'results'],
    'rag_top20': [root / 'rag' / 'results'],
}

filename_map = {
    'oracle': 'oracle_results_eval.json',
    'hipporag2_top5': 'hipporag2_results_top5_eval.json',
    'memoryos': 'memoryos_results_eval.json',
    'rag_top5': 'rag_results_top5_eval.json',
    'rag_top10': 'rag_results_top10_eval.json',
    'rag_top20': 'rag_results_top20_eval.json',
}


def iter_eval_files(baseline: str):
    name = filename_map[baseline]
    for base in baseline_files[baseline]:
        if base.exists():
            yield from base.rglob(f'**/eval/{name}')


def load_llm_scores(path: Path):
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    scores = []
    for sample in data.get('samples', []):
        score = sample.get('scores', {}).get('llm_gpt_score')
        if isinstance(score, (int, float)):
            scores.append(float(score))
    if scores:
        return scores, False
    mean_score = data.get('summary', {}).get('llm_gpt_score_mean')
    if isinstance(mean_score, (int, float)):
        return [float(mean_score)], True
    return [], False


def extract_user_id(path: Path) -> str:
    # expected: .../results/001_user_001/eval/<file>.json
    for part in path.parts:
        if '_user_' in part:
            return part
    return 'unknown_user'


rows = []
for baseline in ['oracle', 'rag_top5', 'rag_top10', 'rag_top20', 'memoryos', 'hipporag2_top5']:
    for fp in sorted(iter_eval_files(baseline)):
        user_id = extract_user_id(fp)
        scores, used_mean = load_llm_scores(fp)
        mean = statistics.mean(scores) if scores else float('nan')
        rows.append({
            'user': user_id,
            'baseline': baseline,
            'mean_llm_score': mean,
            'num_scores': len(scores),
            'used_summary_mean_fallback': used_mean,
            'eval_file': str(fp),
        })
append_rows_to_csv(rows, 'user_level')

try:
    import pandas as pd
    df_user = pd.DataFrame(rows).sort_values(['user', 'baseline'])
    display(df_user)
except Exception:
    rows


## Category Level Scores


In [ ]:
# ---- output capture (shared CSV) ----
from pathlib import Path
import csv

OUTPUT_CSV = Path('eval/statistics_cell_outputs.csv')
CSV_COLUMNS = [
    'section',
    'user',
    'baseline',
    'category',
    'mean_llm_score',
    'num_scores',
    'num_files',
    'used_summary_mean_fallback',
    'eval_file',
]


def append_rows_to_csv(rows, section):
    if not rows:
        return
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    file_exists = OUTPUT_CSV.exists()
    with OUTPUT_CSV.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            record = {k: None for k in CSV_COLUMNS}
            record.update({'section': section})
            record.update(row)
            writer.writerow(record)

from collections import defaultdict
from pathlib import Path
import json
import statistics

root = Path('generation')

baseline_files = {
    'oracle': [root / 'oracle' / 'results'],
    'hipporag2_top5': [root / 'HippoRAG2' / 'results'],
    'hipporag2_top10': [root / 'HippoRAG2' / 'results'],
    'hipporag2_top20': [root / 'HippoRAG2' / 'results'],
    'memoryos': [root / 'MemoryOS' / 'results'],
    'rag_top5': [root / 'rag' / 'results'],
    'rag_top10': [root / 'rag' / 'results'],
    'rag_top20': [root / 'rag' / 'results'],
}

filename_map = {
    'oracle': 'oracle_results_eval.json',
    'hipporag2_top5': 'hipporag2_results_top5_eval.json',
    'hipporag2_top10': 'hipporag2_results_top10_eval.json',
    'hipporag2_top20': 'hipporag2_results_top20_eval.json',
    'memoryos': 'memoryos_results_eval.json',
    'rag_top5': 'rag_results_top5_eval.json',
    'rag_top10': 'rag_results_top10_eval.json',
    'rag_top20': 'rag_results_top20_eval.json',
}


def iter_eval_files(baseline: str):
    name = filename_map[baseline]
    for base in baseline_files[baseline]:
        if base.exists():
            yield from base.rglob(f'**/eval/{name}')


def extract_user_id(path: Path) -> str:
    for part in path.parts:
        if '_user_' in part:
            return part
    return 'unknown_user'


def load_samples(path: Path):
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    return data.get('samples', [])


def get_id(sample):
    raw = sample.get('id')
    if isinstance(raw, int):
        return raw
    if isinstance(raw, str):
        try:
            return int(raw)
        except ValueError:
            return None
    return None


def get_score(sample):
    score = sample.get('scores', {}).get('llm_gpt_score')
    if isinstance(score, (int, float)):
        return float(score)
    return None


CATEGORY_RANGES = [
    (1, 30, 'L2.1'),
    (31, 60, 'L2.2'),
    (61, 90, 'L3.1'),
    (91, 120, 'L3.2'),
    (121, 150, 'L1.1'),
    (151, 180, 'L1.2'),
]


def category_from_id(idx: int) -> str:
    for start, end, label in CATEGORY_RANGES:
        if start <= idx <= end:
            return label
    return 'unknown'


rows = []
for baseline in ['oracle', 'rag_top5', 'rag_top10', 'rag_top20', 'memoryos', 'hipporag2_top5', 'hipporag2_top10', 'hipporag2_top20']:
    for fp in sorted(iter_eval_files(baseline)):
        user_id = extract_user_id(fp)
        cat_scores = defaultdict(list)
        for s in load_samples(fp):
            idx = get_id(s)
            score = get_score(s)
            if idx is None or score is None:
                continue
            cat = category_from_id(idx)
            cat_scores[cat].append(score)
        for cat, scores in sorted(cat_scores.items()):
            mean = statistics.mean(scores) if scores else float('nan')
            rows.append({
                'user': user_id,
                'baseline': baseline,
                'category': cat,
                'mean_llm_score': mean,
                'num_scores': len(scores),
                'eval_file': str(fp),
            })
append_rows_to_csv(rows, 'category_level')

try:
    import pandas as pd
    df_cat = pd.DataFrame(rows).sort_values(['user', 'baseline', 'category'])
    display(df_cat)
except Exception:
    rows

## save category level scores to csv

# ---- output capture (shared CSV) ----
from pathlib import Path
import csv

output_path = Path('eval/category.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

df_cat.to_csv(output_path, index=False)
print(f"Saved results to {output_path}")


## Baseline Level Scores


In [ ]:
# ---- output capture (shared CSV) ----
from pathlib import Path
import csv

OUTPUT_CSV = Path('eval/statistics_cell_outputs.csv')
CSV_COLUMNS = [
    'section',
    'user',
    'baseline',
    'category',
    'mean_llm_score',
    'num_scores',
    'num_files',
    'used_summary_mean_fallback',
    'eval_file',
]


def append_rows_to_csv(rows, section):
    if not rows:
        return
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    file_exists = OUTPUT_CSV.exists()
    with OUTPUT_CSV.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            record = {k: None for k in CSV_COLUMNS}
            record.update({'section': section})
            record.update(row)
            writer.writerow(record)

from collections import defaultdict
from pathlib import Path
import json
import statistics

root = Path('generation')

baseline_files = {
    'oracle': [root / 'oracle' / 'results'],
    'hipporag2_top5': [root / 'HippoRAG2' / 'results'],
    'hipporag2_top10': [root / 'HippoRAG2' / 'results'],
    'hipporag2_top20': [root / 'HippoRAG2' / 'results'],
    'memoryos': [root / 'MemoryOS' / 'results'],
    'rag_top5': [root / 'rag' / 'results'],
    'rag_top10': [root / 'rag' / 'results'],
    'rag_top20': [root / 'rag' / 'results'],
}

filename_map = {
    'oracle': 'oracle_results_eval.json',
    'hipporag2_top5': 'hipporag2_results_top5_eval.json',
    'hipporag2_top10': 'hipporag2_results_top10_eval.json',
    'hipporag2_top20': 'hipporag2_results_top20_eval.json',
    'memoryos': 'memoryos_results_eval.json',
    'rag_top5': 'rag_results_top5_eval.json',
    'rag_top10': 'rag_results_top10_eval.json',
    'rag_top20': 'rag_results_top20_eval.json',
}


def iter_eval_files(baseline: str):
    name = filename_map[baseline]
    for base in baseline_files[baseline]:
        if base.exists():
            yield from base.rglob(f'**/eval/{name}')


def load_samples(path: Path):
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    return data.get('samples', [])


def get_id(sample):
    raw = sample.get('id')
    if isinstance(raw, int):
        return raw
    if isinstance(raw, str):
        try:
            return int(raw)
        except ValueError:
            return None
    return None


def get_score(sample):
    score = sample.get('scores', {}).get('llm_gpt_score')
    if isinstance(score, (int, float)):
        return float(score)
    return None


CATEGORY_RANGES = [
    (1, 30, 'L2.1'),
    (31, 60, 'L2.2'),
    (61, 90, 'L3.1'),
    (91, 120, 'L3.2'),
    (121, 150, 'L1.1'),
    (151, 180, 'L1.2'),
]


def category_from_id(idx: int) -> str:
    for start, end, label in CATEGORY_RANGES:
        if start <= idx <= end:
            return label
    return 'unknown'


rows = []
for baseline in ['oracle', 'rag_top5', 'rag_top10', 'rag_top20', 'memoryos', 'hipporag2_top5', 'hipporag2_top10', 'hipporag2_top20']:
    cat_scores = defaultdict(list)
    for fp in sorted(iter_eval_files(baseline)):
        for s in load_samples(fp):
            idx = get_id(s)
            score = get_score(s)
            if idx is None or score is None:
                continue
            cat = category_from_id(idx)
            cat_scores[cat].append(score)
    for cat, scores in sorted(cat_scores.items()):
        mean = statistics.mean(scores) if scores else float('nan')
        rows.append({
            'baseline': baseline,
            'category': cat,
            'mean_llm_score': mean,
            'num_scores': len(scores),
        })

try:
    import pandas as pd
    df_baseline = pd.DataFrame(rows).sort_values(['baseline', 'category'])
    display(df_baseline)
except Exception:
    rows

output_path = Path('eval/baseline_category.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

df_baseline.to_csv(output_path, index=False)
print(f"Saved results to {output_path}")


## Add Domain Level result

In [ ]:
# ---- output capture (shared CSV) ----
from pathlib import Path
import csv

OUTPUT_CSV = Path('eval/statistics_cell_outputs.csv')
CSV_COLUMNS = [
    'section',
    'user',
    'baseline',
    'category',
    'mean_llm_score',
    'num_scores',
    'num_files',
    'used_summary_mean_fallback',
    'eval_file',
]


def append_rows_to_csv(rows, section):
    if not rows:
        return
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    file_exists = OUTPUT_CSV.exists()
    with OUTPUT_CSV.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            record = {k: None for k in CSV_COLUMNS}
            record.update({'section': section})
            record.update(row)
            writer.writerow(record)

from collections import defaultdict
from pathlib import Path
import json
import re
import statistics

root = Path('generation')
source_root = Path('data_construction/generated_outputs')

baseline_files = {
    'oracle': [root / 'oracle' / 'results'],
    'hipporag2_top5': [root / 'HippoRAG2' / 'results'],
    'hipporag2_top10': [root / 'HippoRAG2' / 'results'],
    'hipporag2_top20': [root / 'HippoRAG2' / 'results'],
    'memoryos': [root / 'MemoryOS' / 'results'],
    'rag_top5': [root / 'rag' / 'results'],
    'rag_top10': [root / 'rag' / 'results'],
    'rag_top20': [root / 'rag' / 'results'],
}

filename_map = {
    'oracle': 'oracle_results_eval.json',
    'hipporag2_top5': 'hipporag2_results_top5_eval.json',
    'hipporag2_top10': 'hipporag2_results_top10_eval.json',
    'hipporag2_top20': 'hipporag2_results_top20_eval.json',
    'memoryos': 'memoryos_results_eval.json',
    'rag_top5': 'rag_results_top5_eval.json',
    'rag_top10': 'rag_results_top10_eval.json',
    'rag_top20': 'rag_results_top20_eval.json',
}


def iter_eval_files(baseline: str):
    name = filename_map[baseline]
    for base in baseline_files[baseline]:
        if base.exists():
            yield from base.rglob(f'**/eval/{name}')


def extract_user_id(path: Path) -> str:
    for part in path.parts:
        if '_user_' in part:
            return part
    return 'unknown_user'


def load_samples(path: Path):
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    return data.get('samples', [])


def get_score(sample):
    score = sample.get('scores', {}).get('llm_gpt_score')
    if isinstance(score, (int, float)):
        return float(score)
    return None


def normalize_domain(value):
    if isinstance(value, str) and value.strip():
        return value.strip()
    if isinstance(value, list):
        items = [str(v).strip() for v in value if str(v).strip()]
        if items:
            return '; '.join(items)
    return None


def slugify(value: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")
    return slug or "domain"


def slug_from_event_id(event_id):
    if not isinstance(event_id, str) or not event_id:
        return None
    if '_w' in event_id:
        event_id = event_id.split('_w', 1)[0]
    return event_id or None


def load_domain_lookup(user_id: str):
    lookup = {}
    path = source_root / user_id / 'app_logs_final.json'
    if not path.exists():
        return lookup
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    slug_map = {}
    for name in data.get('domains', []) or []:
        norm = normalize_domain(name)
        if norm:
            slug_map[slugify(norm)] = norm
    for app_log in data.get('app_logs', []):
        app_log_id = app_log.get('app_log_id')
        domain = normalize_domain(app_log.get('metadata', {}).get('domain'))
        if not domain:
            slug = slug_from_event_id(app_log.get('event_id'))
            if slug:
                domain = slug_map.get(slug)
        if app_log_id and domain:
            lookup[str(app_log_id)] = domain
    return lookup


def domain_from_reference(sample, domain_lookup):
    meta = sample.get('metadata', {}) or {}
    refs = meta.get('reference_evidence') or []
    if isinstance(refs, (list, tuple)) and refs:
        anchor = refs[0]
        if anchor is not None:
            return domain_lookup.get(str(anchor))
    return None


def get_domain(sample, domain_lookup):
    meta = sample.get('metadata', {}) or {}
    direct = normalize_domain(meta.get('belonged') or meta.get('domain'))
    if direct:
        return direct
    ref_domain = domain_from_reference(sample, domain_lookup)
    if ref_domain:
        return ref_domain
    return 'unknown'


user_rows = []
unknown_rows = []
for baseline in ['oracle', 'rag_top5', 'rag_top10', 'rag_top20', 'memoryos', 'hipporag2_top5', 'hipporag2_top10', 'hipporag2_top20']:
    domain_cache = {}
    for fp in sorted(iter_eval_files(baseline)):
        user_id = extract_user_id(fp)
        if user_id not in domain_cache:
            domain_cache[user_id] = load_domain_lookup(user_id)
        domain_lookup = domain_cache[user_id]
        domain_scores = defaultdict(list)
        for s in load_samples(fp):
            score = get_score(s)
            if score is None:
                continue
            domain = get_domain(s, domain_lookup)
            domain_scores[domain].append(score)
            if domain == 'unknown':
                meta = s.get('metadata', {}) or {}
                refs = meta.get('reference_evidence') or []
                anchor = refs[0] if isinstance(refs, (list, tuple)) and refs else None
                unknown_rows.append({
                    'user': user_id,
                    'baseline': baseline,
                    'eval_file': str(fp),
                    'sample_id': s.get('id'),
                    'anchor': anchor,
                    'app_log_hit': str(anchor) in domain_lookup if anchor is not None else False,
                })
        for domain, scores in sorted(domain_scores.items()):
            mean = statistics.mean(scores) if scores else float('nan')
            user_rows.append({
                'user': user_id,
                'baseline': baseline,
                'category': domain,
                'mean_llm_score': mean,
                'num_scores': len(scores),
                'eval_file': str(fp),
            })

append_rows_to_csv(user_rows, 'domain_level_user')

baseline_rows = []
for baseline in ['oracle', 'rag_top5', 'rag_top10', 'rag_top20', 'memoryos', 'hipporag2_top5', 'hipporag2_top10', 'hipporag2_top20']:
    domain_scores = defaultdict(list)
    domain_cache = {}
    for fp in sorted(iter_eval_files(baseline)):
        user_id = extract_user_id(fp)
        if user_id not in domain_cache:
            domain_cache[user_id] = load_domain_lookup(user_id)
        domain_lookup = domain_cache[user_id]
        for s in load_samples(fp):
            score = get_score(s)
            if score is None:
                continue
            domain = get_domain(s, domain_lookup)
            domain_scores[domain].append(score)
    for domain, scores in sorted(domain_scores.items()):
        mean = statistics.mean(scores) if scores else float('nan')
        baseline_rows.append({
            'baseline': baseline,
            'category': domain,
            'mean_llm_score': mean,
            'num_scores': len(scores),
        })

append_rows_to_csv(baseline_rows, 'domain_level_baseline')

import pandas as pd

df_domain_user = pd.DataFrame(user_rows).sort_values(['user', 'baseline', 'category'])
df_domain_base = pd.DataFrame(baseline_rows).sort_values(['baseline', 'category'])

df_unknown = pd.DataFrame(unknown_rows).sort_values(['user', 'baseline'])

try:
    display(df_domain_user)
    display(df_domain_base)
    display(df_unknown.head(50))
except Exception:
    df_domain_user, df_domain_base, df_unknown

user_output = Path('eval/domain_user.csv')
base_output = Path('eval/baseline_domain.csv')
unknown_output = Path('eval/domain_unknown_debug.csv')
user_output.parent.mkdir(parents=True, exist_ok=True)

df_domain_user.to_csv(user_output, index=False)
df_domain_base.to_csv(base_output, index=False)
df_unknown.to_csv(unknown_output, index=False)
print(f"Saved results to {user_output}")
print(f"Saved results to {base_output}")
print(f"Saved unknown debug to {unknown_output}")



In [ ]:
# ---- output capture (shared CSV) ----
from pathlib import Path
import csv

OUTPUT_CSV = Path('eval/statistics_cell_outputs.csv')
CSV_COLUMNS = [
    'section',
    'user',
    'baseline',
    'category',
    'mean_llm_score',
    'num_scores',
    'num_files',
    'used_summary_mean_fallback',
    'eval_file',
]


def append_rows_to_csv(rows, section):
    if not rows:
        return
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    file_exists = OUTPUT_CSV.exists()
    with OUTPUT_CSV.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            record = {k: None for k in CSV_COLUMNS}
            record.update({'section': section})
            record.update(row)
            writer.writerow(record)

from collections import defaultdict
from pathlib import Path
import json
import re
import statistics

root = Path('generation')
source_root = Path('data_construction/generated_outputs')

baseline_files = {
    'oracle': [root / 'oracle' / 'results'],
    'hipporag2_top5': [root / 'HippoRAG2' / 'results'],
    'hipporag2_top10': [root / 'HippoRAG2' / 'results'],
    'hipporag2_top20': [root / 'HippoRAG2' / 'results'],
    'memoryos': [root / 'MemoryOS' / 'results'],
    'rag_top5': [root / 'rag' / 'results'],
    'rag_top10': [root / 'rag' / 'results'],
    'rag_top20': [root / 'rag' / 'results'],
}

filename_map = {
    'oracle': 'oracle_results_eval.json',
    'hipporag2_top5': 'hipporag2_results_top5_eval.json',
    'hipporag2_top10': 'hipporag2_results_top10_eval.json',
    'hipporag2_top20': 'hipporag2_results_top20_eval.json',
    'memoryos': 'memoryos_results_eval.json',
    'rag_top5': 'rag_results_top5_eval.json',
    'rag_top10': 'rag_results_top10_eval.json',
    'rag_top20': 'rag_results_top20_eval.json',
}


def iter_eval_files(baseline: str):
    name = filename_map[baseline]
    for base in baseline_files[baseline]:
        if base.exists():
            yield from base.rglob(f'**/eval/{name}')


def extract_user_id(path: Path) -> str:
    for part in path.parts:
        if '_user_' in part:
            return part
    return 'unknown_user'


def load_samples(path: Path):
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    return data.get('samples', [])


def get_score(sample):
    score = sample.get('scores', {}).get('llm_gpt_score')
    if isinstance(score, (int, float)):
        return float(score)
    return None


def normalize_domain(value):
    if isinstance(value, str) and value.strip():
        return value.strip()
    if isinstance(value, list):
        items = [str(v).strip() for v in value if str(v).strip()]
        if items:
            return '; '.join(items)
    return None


def slugify(value: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")
    return slug or "domain"


def slug_from_event_id(event_id):
    if not isinstance(event_id, str) or not event_id:
        return None
    if '_w' in event_id:
        event_id = event_id.split('_w', 1)[0]
    return event_id or None


def load_domain_lookup(user_id: str):
    lookup = {}
    path = source_root / user_id / 'app_logs_final.json'
    if not path.exists():
        return lookup
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    slug_map = {}
    for name in data.get('domains', []) or []:
        norm = normalize_domain(name)
        if norm:
            slug_map[slugify(norm)] = norm
    for app_log in data.get('app_logs', []):
        app_log_id = app_log.get('app_log_id')
        domain = normalize_domain(app_log.get('metadata', {}).get('domain'))
        if not domain:
            slug = slug_from_event_id(app_log.get('event_id'))
            if slug:
                domain = slug_map.get(slug)
        if app_log_id and domain:
            lookup[str(app_log_id)] = domain
    return lookup


def domain_from_reference(sample, domain_lookup):
    meta = sample.get('metadata', {}) or {}
    refs = meta.get('reference_evidence') or []
    if isinstance(refs, (list, tuple)) and refs:
        anchor = refs[0]
        if anchor is not None:
            return domain_lookup.get(str(anchor))
    return None


def get_domain(sample, domain_lookup):
    meta = sample.get('metadata', {}) or {}
    direct = normalize_domain(meta.get('belonged') or meta.get('domain'))
    if direct:
        return direct
    ref_domain = domain_from_reference(sample, domain_lookup)
    if ref_domain:
        return ref_domain
    return 'unknown'


user_rows = []
unknown_rows = []
for baseline in ['oracle', 'rag_top5', 'rag_top10', 'rag_top20', 'memoryos', 'hipporag2_top5', 'hipporag2_top10', 'hipporag2_top20']:
    domain_cache = {}
    for fp in sorted(iter_eval_files(baseline)):
        user_id = extract_user_id(fp)
        if user_id not in domain_cache:
            domain_cache[user_id] = load_domain_lookup(user_id)
        domain_lookup = domain_cache[user_id]
        domain_scores = defaultdict(list)
        for s in load_samples(fp):
            score = get_score(s)
            if score is None:
                continue
            domain = get_domain(s, domain_lookup)
            domain_scores[domain].append(score)
            if domain == 'unknown':
                meta = s.get('metadata', {}) or {}
                refs = meta.get('reference_evidence') or []
                anchor = refs[0] if isinstance(refs, (list, tuple)) and refs else None
                unknown_rows.append({
                    'user': user_id,
                    'baseline': baseline,
                    'eval_file': str(fp),
                    'sample_id': s.get('id'),
                    'anchor': anchor,
                    'app_log_hit': str(anchor) in domain_lookup if anchor is not None else False,
                })
        for domain, scores in sorted(domain_scores.items()):
            mean = statistics.mean(scores) if scores else float('nan')
            user_rows.append({
                'user': user_id,
                'baseline': baseline,
                'category': domain,
                'mean_llm_score': mean,
                'num_scores': len(scores),
                'eval_file': str(fp),
            })

append_rows_to_csv(user_rows, 'domain_level_user')

baseline_rows = []
for baseline in ['oracle', 'rag_top5', 'rag_top10', 'rag_top20', 'memoryos', 'hipporag2_top5', 'hipporag2_top10', 'hipporag2_top20']:
    domain_scores = defaultdict(list)
    domain_cache = {}
    for fp in sorted(iter_eval_files(baseline)):
        user_id = extract_user_id(fp)
        if user_id not in domain_cache:
            domain_cache[user_id] = load_domain_lookup(user_id)
        domain_lookup = domain_cache[user_id]
        for s in load_samples(fp):
            score = get_score(s)
            if score is None:
                continue
            domain = get_domain(s, domain_lookup)
            domain_scores[domain].append(score)
    for domain, scores in sorted(domain_scores.items()):
        mean = statistics.mean(scores) if scores else float('nan')
        baseline_rows.append({
            'baseline': baseline,
            'category': domain,
            'mean_llm_score': mean,
            'num_scores': len(scores),
        })

append_rows_to_csv(baseline_rows, 'domain_level_baseline')

import pandas as pd

df_domain_user = pd.DataFrame(user_rows).sort_values(['user', 'baseline', 'category'])
df_domain_base = pd.DataFrame(baseline_rows).sort_values(['baseline', 'category'])

df_unknown = pd.DataFrame(unknown_rows).sort_values(['user', 'baseline'])

try:
    display(df_domain_user)
    display(df_domain_base)
    display(df_unknown.head(50))
except Exception:
    df_domain_user, df_domain_base, df_unknown

user_output = Path('eval/domain_user.csv')
base_output = Path('eval/baseline_domain.csv')
unknown_output = Path('eval/domain_unknown_debug.csv')
user_output.parent.mkdir(parents=True, exist_ok=True)

df_domain_user.to_csv(user_output, index=False)
df_domain_base.to_csv(base_output, index=False)
df_unknown.to_csv(unknown_output, index=False)
print(f"Saved results to {user_output}")
print(f"Saved results to {base_output}")
print(f"Saved unknown debug to {unknown_output}")



## Acc. versus. Required # Evidence

In [ ]:
from pathlib import Path
import json
import pandas as pd

# llm_gpt_score: 1 represents 0.1 point
SCORE_SCALE = 10

LEVEL_RANGES = [
    (1, 30, '2.1'),
    (31, 60, '2.2'),
    (61, 90, '3.1'),
    (91, 120, '3.2'),
    (121, 150, '1.1'),
    (151, 180, '1.2'),
]

def level_for_index(index_1based: int) -> str:
    for start, end, level in LEVEL_RANGES:
        if start <= index_1based <= end:
            return level
    return 'unknown'

root = Path('generation')

baseline_files = {
    'oracle': [root / 'oracle' / 'results'],
    'hipporag2_top5': [root / 'HippoRAG2' / 'results'],
    'hipporag2_top10': [root / 'HippoRAG2' / 'results'],
    'hipporag2_top20': [root / 'HippoRAG2' / 'results'],
    'memoryos': [root / 'MemoryOS' / 'results'],
    'rag_top5': [root / 'rag' / 'results'],
    'rag_top10': [root / 'rag' / 'results'],
    'rag_top20': [root / 'rag' / 'results'],
}

filename_map = {
    'oracle': 'oracle_results_eval.json',
    'hipporag2_top5': 'hipporag2_results_top5_eval.json',
    'hipporag2_top10': 'hipporag2_results_top10_eval.json',
    'hipporag2_top20': 'hipporag2_results_top20_eval.json',
    'memoryos': 'memoryos_results_eval.json',
    'rag_top5': 'rag_results_top5_eval.json',
    'rag_top10': 'rag_results_top10_eval.json',
    'rag_top20': 'rag_results_top20_eval.json',
}

def iter_eval_files(baseline: str):
    name = filename_map[baseline]
    for base in baseline_files[baseline]:
        if base.exists():
            yield from base.rglob(f'**/eval/{name}')

def ensure_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(v) for v in value if v]
    return [str(value)]

rows = []

for baseline in ['oracle', 'rag_top5', 'rag_top10', 'rag_top20', 'memoryos', 'hipporag2_top5', 'hipporag2_top10', 'hipporag2_top20']:
# for baseline in ['rag_top5']:
    for fp in sorted(iter_eval_files(baseline)):
        with fp.open('r', encoding='utf-8') as f:
            data = json.load(f)
        for idx, sample in enumerate(data.get('samples', []), start=1):
            level = level_for_index(idx)
            meta = sample.get('metadata') or {}
            if level == '1.2':
                gold_ids = ensure_list(meta.get('reference_evidence'))
            else:
                gold_ids = ensure_list(meta.get('app_log_ids'))
            required_cnt = len(set(gold_ids))
            score = sample.get('scores', {}).get('llm_gpt_score')
            if not isinstance(score, (int, float)):
                continue
            rows.append({
                'baseline': baseline,
                'required_evidence': required_cnt,
                'score': float(score) * SCORE_SCALE,
            })

df = pd.DataFrame(rows)

score_vs_required = (
    df.groupby(['baseline', 'required_evidence'], as_index=False)
      .agg(
          avg_score=('score', 'mean'),
          num_questions=('score', 'size'),
      )
)
score_vs_required['avg_score_1dp'] = score_vs_required['avg_score'].round(1)

display(score_vs_required.sort_values(['baseline', 'required_evidence']))

output_path = Path('eval/score_vs_required_evidence.csv')
score_vs_required.to_csv(output_path, index=False)
print(f'Saved score vs required evidence to {output_path}')
